## Start pyspark

In [1]:
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName("movielens-local-bronze")
    .master("local[*]")
    .config("spark.sql.shuffle.partitions", "8")
    .getOrCreate()
)

spark.sparkContext.setLogLevel("ERROR")


## Define schemas

In [3]:
from pyspark.sql.types import StructType, StructField, IntegerType, DoubleType, LongType,StringType

movies_schema = StructType([
    StructField("movieId", IntegerType(), False),
    StructField("title", StringType(), True),
    StructField("genres", StringType(), True),
])

links_schema = StructType([
    StructField("movieId", IntegerType(), False),
    StructField("imdbId", IntegerType(), True),
    StructField("tmdbId", IntegerType(), True),
])

tags_schema = StructType([
    StructField("userId", IntegerType(), False),
    StructField("movieId", IntegerType(), False),
    StructField("tag", StringType(), True),
    StructField("timestamp", LongType(), True),
])
ratings_schema = StructType([
    StructField("userId", IntegerType(), False),
    StructField("movieId", IntegerType(), False),
    StructField("rating", DoubleType(), True),
    StructField("timestamp", LongType(), True),  # unix epoch seconds
])


## Read data sets

In [4]:
movies_path   = r"..\data\raw\movies.csv"
links_path    = r"..\data\raw\links.csv"
tags_path     = r"..\data\raw\tags.csv"
ratings_path  = r"..\data\raw\ratings.csv"

df_movies_raw  = spark.read.schema(movies_schema).option("header", True).csv(movies_path)
df_links_raw   = spark.read.schema(links_schema).option("header", True).csv(links_path)
df_tags_raw    = spark.read.schema(tags_schema).option("header", True).csv(tags_path)
df_ratings_raw = spark.read.schema(ratings_schema).option("header", True).csv(ratings_path)

print("raw counts:",
      df_movies_raw.count(),
      df_links_raw.count(),
      df_tags_raw.count(),
      df_ratings_raw.count())

df_movies_raw.show(5, truncate=False)
df_links_raw.show(5, truncate=False)
df_tags_raw.show(5, truncate=False)
df_ratings_raw.show(5, truncate=False)

raw counts: 87585 87585 2000072 32000204
+-------+----------------------------------+-------------------------------------------+
|movieId|title                             |genres                                     |
+-------+----------------------------------+-------------------------------------------+
|1      |Toy Story (1995)                  |Adventure|Animation|Children|Comedy|Fantasy|
|2      |Jumanji (1995)                    |Adventure|Children|Fantasy                 |
|3      |Grumpier Old Men (1995)           |Comedy|Romance                             |
|4      |Waiting to Exhale (1995)          |Comedy|Drama|Romance                       |
|5      |Father of the Bride Part II (1995)|Comedy                                     |
+-------+----------------------------------+-------------------------------------------+
only showing top 5 rows

+-------+------+------+
|movieId|imdbId|tmdbId|
+-------+------+------+
|1      |114709|862   |
|2      |113497|8844  |
|3      |1132

## Clean movies (genres array + year + title_clean)

In [5]:
from pyspark.sql import functions as F

df_movies_silver = (
    df_movies_raw
    .dropDuplicates(["movieId"])
    .withColumn("title", F.trim("title"))
    .withColumn("genres", F.trim("genres"))
    .withColumn("genres_array", F.split(F.coalesce("genres", F.lit("")), "\\|"))
    .withColumn("genres_array", F.expr("filter(genres_array, x -> x != '' and x != '(no genres listed)')"))
    .withColumn("movie_year", F.regexp_extract("title", r"\((\d{4})\)\s*$", 1).cast("int"))
    .withColumn("title_clean", F.regexp_replace("title", r"\s*\(\d{4}\)\s*$", ""))
)

df_movies_silver.select("movieId","title","title_clean","movie_year","genres_array").show(5, truncate=False)


+-------+----------------------------------+---------------------------+----------+-----------------------------+
|movieId|title                             |title_clean                |movie_year|genres_array                 |
+-------+----------------------------------+---------------------------+----------+-----------------------------+
|5      |Father of the Bride Part II (1995)|Father of the Bride Part II|1995      |[Comedy]                     |
|6      |Heat (1995)                       |Heat                       |1995      |[Action, Crime, Thriller]    |
|9      |Sudden Death (1995)               |Sudden Death               |1995      |[Action]                     |
|10     |GoldenEye (1995)                  |GoldenEye                  |1995      |[Action, Adventure, Thriller]|
|12     |Dracula: Dead and Loving It (1995)|Dracula: Dead and Loving It|1995      |[Comedy, Horror]             |
+-------+----------------------------------+---------------------------+----------+-----

## Clean links (dedupe)

In [6]:
df_links_silver = df_links_raw.dropDuplicates(["movieId"])
print("links distinct movieId:", df_links_silver.select("movieId").distinct().count())
df_links_silver.show(5, truncate=False)


links distinct movieId: 87585
+-------+------+------+
|movieId|imdbId|tmdbId|
+-------+------+------+
|12     |112896|12110 |
|13     |112453|21032 |
|14     |113987|10858 |
|18     |113101|5     |
|38     |113442|33689 |
+-------+------+------+
only showing top 5 rows



## Clean tags (timestamp + orphan check)

In [7]:
df_tags_silver = (
    df_tags_raw
    .dropDuplicates(["userId","movieId","timestamp","tag"])
    .withColumn("tag_ts", F.from_unixtime("timestamp").cast("timestamp"))
)

orphan_tags = df_tags_silver.join(df_movies_silver.select("movieId"), on="movieId", how="left_anti")
print("orphan tags:", orphan_tags.count())
orphan_tags.show(10, truncate=False)


orphan tags: 0
+-------+------+---+---------+------+
|movieId|userId|tag|timestamp|tag_ts|
+-------+------+---+---------+------+
+-------+------+---+---------+------+



## Clean ratings (timestamp + domain + orphan check)

In [8]:
df_ratings_silver = (
    df_ratings_raw
    .dropDuplicates(["userId","movieId","timestamp"])
    .withColumn("rating_ts", F.from_unixtime("timestamp").cast("timestamp"))
    .filter(F.col("rating").between(0.5, 5.0))
)

orphan_ratings = df_ratings_silver.join(df_movies_silver.select("movieId"), on="movieId", how="left_anti")
print("orphan ratings:", orphan_ratings.count())
orphan_ratings.show(10, truncate=False)

orphan ratings: 0
+-------+------+------+---------+---------+
|movieId|userId|rating|timestamp|rating_ts|
+-------+------+------+---------+---------+
+-------+------+------+---------+---------+



## Build “movie master” (movies + links)

In [14]:
df_movies_master = df_movies_silver.join(df_links_silver, on="movieId", how="left")
print("movie master rows:", df_movies_master.count())
df_movies_master.select("movieId","title_clean","imdbId","tmdbId","genres_array").show(5, truncate=False)


movie master rows: 87585
+-------+---------------------------+------+------+-----------------------------+
|movieId|title_clean                |imdbId|tmdbId|genres_array                 |
+-------+---------------------------+------+------+-----------------------------+
|5      |Father of the Bride Part II|113041|11862 |[Comedy]                     |
|6      |Heat                       |113277|949   |[Action, Crime, Thriller]    |
|9      |Sudden Death               |114576|9091  |[Action]                     |
|10     |GoldenEye                  |113189|710   |[Action, Adventure, Thriller]|
|12     |Dracula: Dead and Loving It|112896|12110 |[Comedy, Horror]             |
+-------+---------------------------+------+------+-----------------------------+
only showing top 5 rows



# Gold transformations

## Performance setup (important with 32M ratings)

In [9]:
from pyspark.sql import functions as F

# Tune partitions for local machine (adjust 8/16/32 based on CPU)
spark.conf.set("spark.sql.shuffle.partitions", "16")

# Cache ratings; we’ll reuse it multiple times
df_ratings_silver = df_ratings_silver.select("userId", "movieId", "rating", "rating_ts").cache()
_ = df_ratings_silver.count()   # materialize cache


## Gold: fact_ratings (ALS-ready)

In [10]:
fact_ratings = (
    df_ratings_silver
    .select("userId", "movieId", "rating")
)

print("fact_ratings rows:", fact_ratings.count())
fact_ratings.show(5)


fact_ratings rows: 32000204
+------+-------+------+
|userId|movieId|rating|
+------+-------+------+
|     1|   1939|   5.0|
|     1|   3078|   2.0|
|     2|     31|   5.0|
|     2|     34|   5.0|
|     2|    207|   5.0|
+------+-------+------+
only showing top 5 rows



## dev sampling for dim_users

In [11]:
from pyspark.sql import functions as F

# 1% sample for local dev (tune 0.005–0.05 depending on laptop)
ratings_dev = df_ratings_silver.sample(False, 0.01, seed=42).cache()
_ = ratings_dev.count()


## Gold: dim_users (behavior profile)

In [12]:
dim_users_dev = (
    ratings_dev
    .groupBy("userId")
    .agg(
        F.count("*").alias("num_ratings"),
        F.avg("rating").alias("avg_rating"),
        F.min("rating_ts").alias("first_rating_ts"),
        F.max("rating_ts").alias("last_rating_ts"),
        F.countDistinct("movieId").alias("distinct_movies_rated"),
    )
    .withColumn("is_power_user", F.col("num_ratings") >= F.lit(200))
)
dim_users_dev.orderBy(F.desc("num_ratings")).limit(10).show(truncate=False)

+------+-----------+------------------+-------------------+-------------------+---------------------+-------------+
|userId|num_ratings|avg_rating        |first_rating_ts    |last_rating_ts     |distinct_movies_rated|is_power_user|
+------+-----------+------------------+-------------------+-------------------+---------------------+-------------+
|175325|341        |3.095307917888563 |2015-12-15 07:59:34|2020-06-12 17:34:55|341                  |true         |
|171795|109        |3.123853211009174 |2009-04-07 05:28:47|2019-08-23 02:07:18|109                  |false        |
|17035 |93         |2.403225806451613 |2015-07-06 00:27:27|2023-06-28 08:07:38|93                   |false        |
|55653 |90         |3.327777777777778 |2001-08-05 20:19:38|2014-09-28 04:08:52|90                   |false        |
|123465|89         |2.4887640449438204|2010-11-28 03:06:13|2023-05-15 21:50:13|89                   |false        |
|10202 |81         |3.388888888888889 |2001-09-10 17:37:06|2023-01-21 01

## Movie rating stats (use the same ratings_dev sample)

In [14]:
from pyspark.sql import functions as F

movie_rating_stats_dev = (
    ratings_dev
    .groupBy("movieId")
    .agg(
        F.count("*").alias("num_ratings"),
        F.avg("rating").alias("avg_rating")
    )
)

movie_rating_stats_dev.orderBy(F.desc("num_ratings")).show(10, truncate=False)


+-------+-----------+------------------+
|movieId|num_ratings|avg_rating        |
+-------+-----------+------------------+
|318    |1064       |4.404605263157895 |
|296    |1012       |4.1729249011857705|
|356    |952        |4.046743697478991 |
|2571   |937        |4.107790821771611 |
|593    |897        |4.12876254180602  |
|260    |866        |4.040993071593533 |
|480    |791        |3.684576485461441 |
|2959   |750        |4.212             |
|7153   |738        |4.053523035230352 |
|1196   |737        |4.113297150610584 |
+-------+-----------+------------------+
only showing top 10 rows



## Build dim_movies_enriched (movies + links + rating stats)

In [15]:
df_movies_master = df_movies_silver.join(df_links_silver, on="movieId", how="left")

dim_movies_enriched_dev = (
    df_movies_master
    .join(movie_rating_stats_dev, on="movieId", how="left")
    .fillna({"num_ratings": 0})
)

dim_movies_enriched_dev.select(
    "movieId", "title_clean", "movie_year", "genres_array", "imdbId", "tmdbId", "num_ratings", "avg_rating"
).orderBy(F.desc("num_ratings")).show(10, truncate=False)


+-------+----------------------------------------------+----------+-------------------------------------+------+------+-----------+------------------+
|movieId|title_clean                                   |movie_year|genres_array                         |imdbId|tmdbId|num_ratings|avg_rating        |
+-------+----------------------------------------------+----------+-------------------------------------+------+------+-----------+------------------+
|318    |Shawshank Redemption, The                     |1994      |[Crime, Drama]                       |111161|278   |1064       |4.404605263157895 |
|296    |Pulp Fiction                                  |1994      |[Comedy, Crime, Drama, Thriller]     |110912|680   |1012       |4.1729249011857705|
|356    |Forrest Gump                                  |1994      |[Comedy, Drama, Romance, War]        |109830|13    |952        |4.046743697478991 |
|2571   |Matrix, The                                   |1999      |[Action, Sci-Fi, Thriller] 

In [16]:
top_movies = (
    dim_movies_enriched_dev
    .orderBy(F.desc("num_ratings"))
    .select("movieId", "title_clean", "movie_year", "imdbId", "tmdbId", "num_ratings")
    .limit(500)
)

top_movies.show(20, truncate=False)



+-------+-----------------------------------------------------------------------+----------+------+------+-----------+
|movieId|title_clean                                                            |movie_year|imdbId|tmdbId|num_ratings|
+-------+-----------------------------------------------------------------------+----------+------+------+-----------+
|318    |Shawshank Redemption, The                                              |1994      |111161|278   |1064       |
|296    |Pulp Fiction                                                           |1994      |110912|680   |1012       |
|356    |Forrest Gump                                                           |1994      |109830|13    |952        |
|2571   |Matrix, The                                                            |1999      |133093|603   |937        |
|593    |Silence of the Lambs, The                                              |1991      |102926|274   |897        |
|260    |Star Wars: Episode IV - A New Hope     

In [17]:
top500_pairs = [
    (int(r["movieId"]), int(r["imdbId"]))
    for r in top_movies.select("movieId", "imdbId").collect()
    if r["imdbId"] is not None
]

len(top500_pairs), top500_pairs[:5]

(500,
 [(318, 111161), (296, 110912), (356, 109830), (2571, 133093), (593, 102926)])

## scraping

In [18]:
# If not installed yet (run once):
# %pip install requests beautifulsoup4 lxml tenacity

import json
import re
import time
import requests
from bs4 import BeautifulSoup
from tenacity import retry, stop_after_attempt, wait_exponential, retry_if_exception_type

from pyspark.sql import functions as F


## Select top 5 movies to scrape

In [26]:
top_movies_5 = (
    dim_movies_enriched_dev
    .orderBy(F.desc("num_ratings"))
    .select("movieId", "title_clean", "movie_year", "imdbId", "tmdbId", "num_ratings")
    .limit(500)
    .cache()
)

print("Top 500 preview:")
top_movies_5.show(truncate=False)

top5_pairs = [
    (int(r["movieId"]), int(r["imdbId"]))
    for r in top_movies_5.select("movieId", "imdbId").collect()
    if r["imdbId"] is not None
]

print("Pairs (movieId, imdbId):", top5_pairs)


Top 500 preview:
+-------+-----------------------------------------------------------------------+----------+------+------+-----------+
|movieId|title_clean                                                            |movie_year|imdbId|tmdbId|num_ratings|
+-------+-----------------------------------------------------------------------+----------+------+------+-----------+
|318    |Shawshank Redemption, The                                              |1994      |111161|278   |1064       |
|296    |Pulp Fiction                                                           |1994      |110912|680   |1012       |
|356    |Forrest Gump                                                           |1994      |109830|13    |952        |
|2571   |Matrix, The                                                            |1999      |133093|603   |937        |
|593    |Silence of the Lambs, The                                              |1991      |102926|274   |897        |
|260    |Star Wars: Episode IV 

## IMDb scraper (director/title/poster from JSON-LD + budget)

In [27]:
SESSION = requests.Session()
HEADERS = {"User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64)"}

_money_re = re.compile(r"\$[\d,]+")

@retry(
    reraise=True,
    stop=stop_after_attempt(3),
    wait=wait_exponential(multiplier=1, min=2, max=10),
    retry=retry_if_exception_type(requests.RequestException),
)
def fetch_imdb_html(imdb_id: int) -> str:
    imdb7 = str(imdb_id).zfill(7)
    url = f"https://www.imdb.com/title/tt{imdb7}/"
    resp = SESSION.get(url, headers=HEADERS, timeout=25)
    resp.raise_for_status()
    return resp.text

def parse_jsonld(soup: BeautifulSoup) -> dict:
    script = soup.find("script", type="application/ld+json")
    if not script or not script.string:
        return {}
    try:
        data = json.loads(script.string)
        return data if isinstance(data, dict) else {}
    except json.JSONDecodeError:
        return {}

def extract_budget_best_effort(soup: BeautifulSoup):
    """
    Tries to find 'Budget' label and grab a nearby '$x,xxx,xxx' token.
    This is heuristic and may return None frequently.
    """
    node = soup.find(string=re.compile(r"^\s*Budget\s*$", re.IGNORECASE))
    if not node:
        return None

    # Walk up a few ancestors and search for $ amounts
    container = node.parent
    for _ in range(6):
        if container is None:
            break
        text = container.get_text(" ", strip=True)
        m = _money_re.search(text)
        if m:
            return m.group(0)
        container = container.parent

    return None

def scrape_imdb_movie(movie_id: int, imdb_id: int) -> dict:
    out = {
        "movieId": int(movie_id),
        "imdbId": int(imdb_id),
        "title": None,
        "director": None,
        "budget": None,        # string like "$50,000,000" if found
        "poster_url": None,
        "status": "ok",
        "source": "imdb"
    }

    try:
        html = fetch_imdb_html(imdb_id)
        soup = BeautifulSoup(html, "lxml")

        # JSON-LD: title, director, poster
        data = parse_jsonld(soup)

        out["title"] = data.get("name")

        director = data.get("director")
        if isinstance(director, dict):
            out["director"] = director.get("name")
        elif isinstance(director, list) and director:
            out["director"] = director[0].get("name")

        image = data.get("image")
        if isinstance(image, str):
            out["poster_url"] = image

        # Budget: heuristic
        out["budget"] = extract_budget_best_effort(soup)

    except requests.HTTPError as e:
        code = e.response.status_code if e.response is not None else "na"
        out["status"] = f"http:{code}"
    except Exception as e:
        out["status"] = f"err:{type(e).__name__}"

    return out


## Scrape top 5

In [28]:
results = []
sleep_sec = 1.0  # be polite; reduce risk of blocks

for i, (movie_id, imdb_id) in enumerate(top5_pairs, start=1):
    rec = scrape_imdb_movie(movie_id, imdb_id)
    results.append(rec)
    print(f"{i}/500 -> {rec['movieId']} status={rec['status']} director={rec['director']} budget={rec['budget']}")
    time.sleep(sleep_sec)

results


1/500 -> 318 status=ok director=Frank Darabont budget=$25,000,000
2/500 -> 296 status=ok director=Quentin Tarantino budget=$8,000,000
3/500 -> 356 status=ok director=Robert Zemeckis budget=$55,000,000
4/500 -> 2571 status=ok director=Lana Wachowski budget=$63,000,000
5/500 -> 593 status=ok director=Jonathan Demme budget=$19,000,000
6/500 -> 260 status=ok director=George Lucas budget=$11,000,000
7/500 -> 480 status=ok director=Steven Spielberg budget=$63,000,000
8/500 -> 2959 status=ok director=David Fincher budget=$63,000,000
9/500 -> 7153 status=ok director=Peter Jackson budget=$94,000,000
10/500 -> 1196 status=ok director=Irvin Kershner budget=$18,000,000
11/500 -> 50 status=ok director=Bryan Singer budget=$6,000,000
12/500 -> 2858 status=ok director=Sam Mendes budget=$15,000,000
13/500 -> 4993 status=ok director=Peter Jackson budget=$93,000,000
14/500 -> 527 status=ok director=Steven Spielberg budget=$22,000,000
15/500 -> 1198 status=ok director=Steven Spielberg budget=$18,000,000
1

[{'movieId': 318,
  'imdbId': 111161,
  'title': 'The Shawshank Redemption',
  'director': 'Frank Darabont',
  'budget': '$25,000,000',
  'poster_url': 'https://m.media-amazon.com/images/M/MV5BMDAyY2FhYjctNDc5OS00MDNlLThiMGUtY2UxYWVkNGY2ZjljXkEyXkFqcGc@._V1_.jpg',
  'status': 'ok',
  'source': 'imdb'},
 {'movieId': 296,
  'imdbId': 110912,
  'title': 'Pulp Fiction',
  'director': 'Quentin Tarantino',
  'budget': '$8,000,000',
  'poster_url': 'https://m.media-amazon.com/images/M/MV5BYTViYTE3ZGQtNDBlMC00ZTAyLTkyODMtZGRiZDg0MjA2YThkXkEyXkFqcGc@._V1_.jpg',
  'status': 'ok',
  'source': 'imdb'},
 {'movieId': 356,
  'imdbId': 109830,
  'title': 'Forrest Gump',
  'director': 'Robert Zemeckis',
  'budget': '$55,000,000',
  'poster_url': 'https://m.media-amazon.com/images/M/MV5BMzZhZTZlMmItNDQ1Yi00NDFlLTlmMGUtNGY2YjkyMWUzZTNiXkEyXkFqcGc@._V1_.jpg',
  'status': 'ok',
  'source': 'imdb'},
 {'movieId': 2571,
  'imdbId': 133093,
  'title': 'The Matrix',
  'director': 'Lana Wachowski',
  'budget': '

## Save to JSONL

In [29]:
out_path = "scraped_metadata_top5.jsonl"

with open(out_path, "w", encoding="utf-8") as f:
    for r in results:
        f.write(json.dumps(r) + "\n")

print("✅ saved:", out_path)


✅ saved: scraped_metadata_top5.jsonl


## Load the scraped JSONL back

In [31]:
scraped_top5_df = spark.read.json("scraped_metadata_top500.jsonl")
scraped_top5_df.show(truncate=False)


+-----------+--------------------+------+-------+-------------------------------------------------------------------------------------------------------------+------+------+-------------------------------------------------+
|budget     |director            |imdbId|movieId|poster_url                                                                                                   |source|status|title                                            |
+-----------+--------------------+------+-------+-------------------------------------------------------------------------------------------------------------+------+------+-------------------------------------------------+
|$25,000,000|Frank Darabont      |111161|318    |https://m.media-amazon.com/images/M/MV5BMDAyY2FhYjctNDc5OS00MDNlLThiMGUtY2UxYWVkNGY2ZjljXkEyXkFqcGc@._V1_.jpg|imdb  |ok    |The Shawshank Redemption                         |
|$8,000,000 |Quentin Tarantino   |110912|296    |https://m.media-amazon.com/images/M/MV5BYTViYTE3ZGQtNDB

## Recreate dim_movies_enriched_dev

In [32]:
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, IntegerType, StringType, LongType, DoubleType

# schemas (ensure these match yours)
movies_schema = StructType([
    StructField("movieId", IntegerType(), False),
    StructField("title", StringType(), True),
    StructField("genres", StringType(), True),
])
links_schema = StructType([
    StructField("movieId", IntegerType(), False),
    StructField("imdbId", IntegerType(), True),
    StructField("tmdbId", IntegerType(), True),
])
ratings_schema = StructType([
    StructField("userId", IntegerType(), False),
    StructField("movieId", IntegerType(), False),
    StructField("rating", DoubleType(), True),
    StructField("timestamp", LongType(), True),
])

movies_path   = r"..\data\raw\movies.csv"
links_path    = r"..\data\raw\links.csv"
ratings_path  = r"..\data\raw\ratings.csv"

df_movies_raw  = spark.read.schema(movies_schema).option("header", True).csv(movies_path)
df_links_raw   = spark.read.schema(links_schema).option("header", True).csv(links_path)
df_ratings_raw = spark.read.schema(ratings_schema).option("header", True).csv(ratings_path)

df_movies_silver = (
    df_movies_raw.dropDuplicates(["movieId"])
    .withColumn("title", F.trim("title"))
    .withColumn("genres", F.trim("genres"))
    .withColumn("genres_array", F.split(F.coalesce("genres", F.lit("")), "\\|"))
    .withColumn("genres_array", F.expr("filter(genres_array, x -> x != '' and x != '(no genres listed)')"))
    .withColumn("movie_year", F.regexp_extract("title", r"\((\d{4})\)\s*$", 1).cast("int"))
    .withColumn("title_clean", F.regexp_replace("title", r"\s*\(\d{4}\)\s*$", ""))
)

df_links_silver = df_links_raw.dropDuplicates(["movieId"])
df_movies_master = df_movies_silver.join(df_links_silver, on="movieId", how="left")

ratings_dev = (
    df_ratings_raw
    .select("movieId", "rating")
    .sample(False, 0.01, seed=42)
    .cache()
)
_ = ratings_dev.count()

movie_stats_dev = (
    ratings_dev.groupBy("movieId")
    .agg(
        F.count("*").alias("num_ratings"),
        F.avg("rating").alias("avg_rating")
    )
)

dim_movies_enriched_dev = (
    df_movies_master
    .join(movie_stats_dev, on="movieId", how="left")
    .fillna({"num_ratings": 0})
)


## fixed join

In [34]:
from pyspark.sql import functions as F

scraped_top5_df2 = scraped_top5_df.select(
    "movieId",
    F.col("title").alias("scraped_title"),
    "director",
    "budget",
    "poster_url",
    "status"
)

movies_enriched_top5 = dim_movies_enriched_dev.join(scraped_top5_df2, on="movieId", how="left")

movies_enriched_top5.select(
    "movieId", "title_clean", "scraped_title", "director", "budget", "poster_url", "status"
).orderBy(F.desc("num_ratings")).show(20, truncate=False)


+-------+-----------------------------------------------------------------------+-------------------------------------------------+--------------------+------------+-------------------------------------------------------------------------------------------------------------+------+
|movieId|title_clean                                                            |scraped_title                                    |director            |budget      |poster_url                                                                                                   |status|
+-------+-----------------------------------------------------------------------+-------------------------------------------------+--------------------+------------+-------------------------------------------------------------------------------------------------------------+------+
|318    |Shawshank Redemption, The                                              |The Shawshank Redemption                         |Frank Darabont      